In [43]:
from pathlib import Path
import pandas as pd
import argparse
import json

In [45]:
# Lists of relevant ICD9 and 10 codes for immunosuppressed patients. Taken from supplementary material of reference study.
ICD9_IMMUNO = []
ICD10_IMMUNO = []

In [46]:
# Loads tables as DataFrames
print('> Loading tables...', end='\r')
df_icu = pd.read_csv(PATH_MIMICIV / 'icustays.csv.gz')
df_patients = pd.read_csv(PATH_MIMICIV / 'patients.csv.gz')
df_icd = pd.read_csv(PATH_MIMICIV / 'diagnoses_icd.csv.gz')
df_dicd = pd.read_csv(PATH_MIMICIV / 'd_icd_diagnoses.csv.gz')

In [47]:
df_icd.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [48]:
df_icd.loc[df_icd.icd_code == '1937']

,subject_id,hadm_id,seq_num,icd_code,icd_version


In [59]:
df_icd9 = df_icd.loc[df_icd.icd_version == 9]
df_icd10 = df_icd.loc[df_icd.icd_version == 10]

In [61]:
set(df_icd.loc[df_icd.icd_code.str.startswith('140')].icd_code.to_list())

{'1400', '1401', '1404', '1409'}

In [50]:
# Load ICD codes as specified in https://link.springer.com/article/10.1186/s40001-025-02622-3
with open(FPATH_JSON_IMMUNO) as f:
    immuno_icds = json.load(f)

    ICD9_IMMUNO = immuno_icds['icd9']
    ICD10_IMMUNO = immuno_icds['icd10']

In [51]:
# Original ICD list contains codes that does not refer to a specific condition,
#   but a category containing such specifications. Since ICD codes are always fully specified in MIMIC-IV
#   (ie, no broad labels are assigned), we need to find the specific subcategories from the original broader specification.
#   We do that by searching for suffixes for each initial code within MIMIC-IV, and keeping all the matches.

# Using sets to ensure unique codes
icd9_full = set()
for x, i in enumerate(ICD9_IMMUNO):
    print(f'> Processing code {i} - ({x+1} out of {len(ICD9_IMMUNO)})' + ' '*50, end='\r')
    icd9_full = icd9_full | set(df_icd.loc[df_icd.icd_code.str.startswith(i)].icd_code.to_list())
    break

icd10_full = set()
for x, i in enumerate(ICD10_IMMUNO):
    print(f'> Processing code {i} - ({x+1} out of {len(ICD10_IMMUNO)})' + ' '*50, end='\r')
    icd10_full = icd10_full | set(df_icd.loc[df_icd.icd_code.str.startswith(i)].icd_code.to_list())
    break
# Export result to disk (same location as the input jsons)


In [52]:
icd9_full

{'1624'}

In [53]:
print(ICD9_IMMUNO)
print()
print(ICD10_IMMUNO)
print(df_icu.columns)
print(df_icd.columns)

['1624', '1984', '1478', '20048', '20402', '1542', '1579', '1578', '1953', '1441', '20037', '20280', '2881', '27949', '1560', '27903', '1533', '20008', '1940', '1961', '20121', '1622', '1992', '20153', '20530', '1573', '1982', '2882', '1760', '1705', '20287', '1974', '1529', '1910', '20412', '28804', '20045', '20211', '1801', '20210', '1929', '1763', '1531', '20050', '20030', '1418', '20047', '20032', '27904', '1832', '20061', '20510', 'V427', '1745', '1461', '1500', '1882', '1887', '1522', '1599', '20058', '19882', '1909', '2880', '1588', '1638', '1888', '1459', '1958', '1943', '1990', '1913', '1700', '1532', '20281', '1631', '20782', '1602', '1600', '1625', '20078', '20020', '1712', '1880', '20005', '20168', '20204', '1908', '27902', '1505', '20080', '1724', '1430', '20591', '20501', '1530', '1821', '1649', '1729', '1706', '20073', '1748', '1479', '1469', '1762', '20010', '1537', '1944', '20273', '1490', '20531', '1515', '1512', '1620', '20051', '20014', '20082', '1890', '1709', '200

In [54]:
df_icu.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
2,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
3,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113
4,10001725,25563031,31205490,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2110-04-11 15:52:22,2110-04-12 23:59:56,1.338588


In [55]:
# Fetch HADM_IDs consistent with immunosuppressed ICDs

# Ensures correct datatype prior to comparison

# Selects relevant entries of immunosuppressed ICD codes
# Treats ICD9 and 10 separately to avoid collisions
df_icd9 = df_icd.loc[df_icd.icd_version == 9]
df_icd10 = df_icd.loc[df_icd.icd_version == 10]

df_hadmid_immuno = pd.concat([
    df_icd9.loc[df_icd9.astype({'icd_code': 'str'}).icd_code.isin(ICD9_IMMUNO)].hadm_id,
    df_icd10.loc[df_icd10.astype({'icd_code': 'str'}).icd_code.isin(ICD10_IMMUNO)].hadm_id
])

# Gets unique IDs
hadmid_immuno = df_hadmid_immuno.unique()

len(hadmid_immuno)

85969

In [56]:
df_icu.size

585448

In [57]:
# Filters by ICU stays
df_icu.loc[df_icu.hadm_id.isin(hadmid_immuno)].stay_id.unique().size

17460